# gmx_correlation Tutorial

**Version 1.2.0** | License: GPL-3.0-or-later

This notebook demonstrates post-processing of `gmx_correlation` output using the
pre-computed matrices in `test_files/`. No GROMACS installation is required.

## What we cover

1. Reading correlation matrices
2. Heatmap visualisation
3. Comparing KSG, Gaussian, and GPU outputs
4. Network construction and community detection
5. Eigenvector centrality and residue ranking
6. Exporting a PyMOL visualisation script

## Test system

The test files contain a 159-residue protein (XSKP1) with:
- `CPU.dat`  — KSG k-NN MI matrix, computed on CPU
- `GPU.dat`  — KSG k-NN MI matrix, computed on Metal GPU
- `LCPU.dat` — Gaussian (linearised) MI matrix
- `CA-XSKP1.pdb` — Cα-only structure for PyMOL export


## Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Point to the analysis script
repo_root = Path('..').resolve()
sys.path.insert(0, str(repo_root / 'bin'))
from analyze_matrix import (
    read_matrix, plot_heatmap, auto_threshold,
    build_graph, detect_communities, compute_centrality,
    network_summary, plot_network, write_pml
)

DATA = repo_root.parent / 'test_files'
OUT  = Path('output')
OUT.mkdir(exist_ok=True)

print('Data directory:', DATA)
print('Files:', [f.name for f in DATA.glob('*.dat')])

## 1. Reading matrices

In [ ]:
cpu  = read_matrix(str(DATA / 'CPU.dat'))
gpu  = read_matrix(str(DATA / 'GPU.dat'))
lcpu = read_matrix(str(DATA / 'LCPU.dat'))

n = cpu.shape[0]
print(f'Matrix size: {n} x {n}')
print(f'CPU  — min={cpu[cpu < 1999].min():.4f}  max={cpu[cpu < 1999].max():.4f}')
print(f'GPU  — min={gpu[gpu < 1999].min():.4f}  max={gpu[gpu < 1999].max():.4f}')
print(f'LCPU — min={lcpu[lcpu < 1999].min():.4f}  max={lcpu[lcpu < 1999].max():.4f}')

## 2. Heatmaps

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, mat, title in zip(axes,
                          [cpu, gpu, lcpu],
                          ['KSG CPU', 'KSG GPU (Metal)', 'Gaussian (CPU)']):
    display = mat.copy().astype(float)
    np.fill_diagonal(display, np.nan)
    vmax = np.nanpercentile(np.abs(display), 99)
    im = ax.imshow(display, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                   origin='upper', aspect='auto', interpolation='nearest')
    plt.colorbar(im, ax=ax, fraction=0.046)
    ax.set_title(title, fontsize=13)
    ax.set_xlabel('Residue')
    ax.set_ylabel('Residue')

plt.suptitle('Correlation matrices — XSKP1 (159 residues)', fontsize=14)
plt.tight_layout()
plt.savefig(OUT / 'heatmaps_comparison.png', dpi=150)
plt.show()
print('Saved heatmaps_comparison.png')

## 3. CPU vs GPU agreement

In [ ]:
# Exclude diagonal sentinel (2000)
mask = ~np.eye(n, dtype=bool)
diff = cpu[mask] - gpu[mask]

print(f'CPU vs GPU  |max diff| = {np.abs(diff).max():.6f}')
print(f'            mean diff  = {diff.mean():.6f}')
print(f'            std  diff  = {diff.std():.6f}')

fig, ax = plt.subplots(figsize=(5, 4))
ax.hist(diff, bins=60, edgecolor='k', linewidth=0.3)
ax.set_xlabel('CPU − GPU (nats)')
ax.set_ylabel('Count')
ax.set_title('KSG CPU vs Metal GPU deviation')
plt.tight_layout()
plt.savefig(OUT / 'cpu_gpu_diff.png', dpi=150)
plt.show()

## 4. Network construction

We use the CPU KSG matrix. The `auto_threshold()` function sets a sensible
default based on the distribution of off-diagonal values.

In [ ]:
thr = auto_threshold(cpu, asymmetric=False)
print(f'Auto threshold: {thr:.4f}')

g = build_graph(cpu, threshold=thr, asymmetric=False)
print(f'Graph: {g.vcount()} nodes, {g.ecount()} edges')

## 5. Community detection

In [ ]:
cl, modularity = detect_communities(g, method='leading_eigenvector', min_size=3)
membership = cl.membership
n_comm = max(membership)

print(f'Communities: {n_comm}  |  Modularity Q = {modularity:.3f}')

# Community size distribution
from collections import Counter
sizes = Counter(membership)
print('\nCommunity sizes:')
for cid in sorted(sizes):
    label = f'comm {cid}' if cid > 0 else 'noise'
    print(f'  {label:10s}: {sizes[cid]:3d} residues')

## 6. Centrality measures

In [ ]:
centrality = compute_centrality(g)
evc = centrality['evc']

# Top 10 residues by eigenvector centrality
top10 = np.argsort(evc)[::-1][:10]
print('Top 10 residues by eigenvector centrality:')
print(f'{"Residue":>8}  {"EVC":>8}  {"Community":>10}  {"Degree":>7}')
for i in top10:
    print(f'{i+1:>8}  {evc[i]:>8.4f}  {membership[i]:>10}  {centrality["degree"][i]:>7}')

In [ ]:
# Save CSV
network_summary(g, membership, centrality, out_csv=str(OUT / 'cpu_network.csv'))

# Network diagram
plot_network(g, membership, centrality, out_path=str(OUT / 'cpu_network.png'))

img = plt.imread(str(OUT / 'cpu_network.png'))
fig, ax = plt.subplots(figsize=(9, 8))
ax.imshow(img)
ax.axis('off')
plt.tight_layout()
plt.show()

## 7. PyMOL export

The cell below writes a `.pml` script. Open it in PyMOL:
```bash
pymol output/cpu_network.pml
```

In [ ]:
pdb_path = str(DATA / 'CA-XSKP1.pdb')

write_pml(
    g, membership, centrality,
    pdb_path=pdb_path,
    out_pml=str(OUT / 'cpu_network.pml'),
    edge_threshold_pct=0.90,
    chain='A'
)

# Preview first 30 lines
with open(OUT / 'cpu_network.pml') as f:
    lines = f.readlines()
print(f'Total lines in .pml: {len(lines)}')
print(''.join(lines[:30]))

## 8. Gaussian vs KSG comparison

The Gaussian (linear) estimator assumes Gaussian-distributed motions.
Compare it to KSG which makes no distributional assumption.

In [ ]:
mask = ~np.eye(n, dtype=bool)

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(cpu[mask], lcpu[mask], s=0.5, alpha=0.3, c='steelblue')
lim = max(cpu[mask].max(), lcpu[mask].max())
ax.plot([0, lim], [0, lim], 'r--', lw=1, label='identity')
ax.set_xlabel('KSG r(MI) (CPU)')
ax.set_ylabel('Gaussian r(MI)')
ax.set_title('KSG vs Gaussian correlation coefficients')
ax.legend()
plt.tight_layout()
plt.savefig(OUT / 'ksg_vs_gaussian.png', dpi=150)
plt.show()

corr = np.corrcoef(cpu[mask], lcpu[mask])[0, 1]
print(f'Pearson r between KSG and Gaussian r(MI): {corr:.4f}')

## Next steps

- Run `gmx_correlation` with `-gcmi` for the Gaussian Copula MI estimator
- Run with `-te -ote te.dat` for transfer entropy (directed network)
- Pass `-gpu` to accelerate on Apple Silicon or NVIDIA GPUs
- Use `gmx_analyze_matrix` on the command line for batch processing

See `README.md` for full option reference and `CITATION.cff` to cite this tool.